# Gather gene annotations produced by each AMG prediction tool

Using annotations produced when running CheckAMG, DRAM-V, and VIBRANT in the notebook `amg_benchmark_predictions.ipynb`. Including annotations/hits that didn't end up in each tool's final/filtered outputs.

In [1]:
! pip install polars --quiet

In [2]:
from pathlib import Path
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks")
GENOME_DIR = ROOT_DIR.joinpath("benchmark_genomes")
MAIN_DIR = ROOT_DIR.joinpath("metabolism_benchmark")

## Load gene predictions

Construct polars LazyFrames but do not collect right away because of their size.

In [3]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [4]:
gene_id_mapping = pl.read_parquet(MAIN_DIR.joinpath("dramv_genes_reformatted/gene_id_mapping.parquet"))

In [5]:
def reformat_for_merge(df: pl.DataFrame) -> pl.DataFrame:
    return df.drop(["subset", "old_gene", "old_scaffold", "metadata"]).rename({
        "new_gene": "gene",
        "new_scaffold": "scaffold",
    })

### From CheckAMG annotate

Requires CheckAMG annotate to have been run on each sample with the flag `--keep-full-hmm-results`.

In [6]:
CHECKAMG_DIR = MAIN_DIR.joinpath("checkamg_annotate_v1.1_outputs")

In [7]:
pattern_checkamg_all_annots = "*/*/wdir/hmm_results.parquet"
paths_checkamg_all_annots = sorted(CHECKAMG_DIR.glob(pattern_checkamg_all_annots))

dfs_checkamg_all_annots = []

for p in paths_checkamg_all_annots:
    parts = p.parts
    try:
        i = parts.index(CHECKAMG_DIR.name)
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
    except (ValueError, IndexError):
        continue

    df = pl.scan_parquet(p)

    df = df.with_columns([
        pl.lit(sample).alias("sample"),
        pl.lit(source).alias("source"),
        pl.lit(ecosystem).alias("ecosystem"),
    ])

    # Add sample prefixes to gene names since some scaffold names are identical across samples
    df = df.with_columns(
        pl.col("sequence").alias("gene_orig"),
        pl.concat_str([pl.col("sample"), pl.lit("__"), pl.col("sequence")]).alias("sequence"),
    )

    df = df.join(
        reformat_for_merge(gene_id_mapping).lazy(),
        left_on=["sequence", "sample"],
        right_on=["gene", "sample"],
        how="left"
    ).rename({"sequence": "gene"})

    dfs_checkamg_all_annots.append(df)

In [8]:
checkamg_hmm_id_to_name = pl.scan_csv(
    "../CheckAMG/files/hmm_id_to_name.tsv",
    separator="\t"
)

In [9]:
checkamg_all_annots = (
    pl.concat(dfs_checkamg_all_annots, how="diagonal_relaxed")
    .join(checkamg_hmm_id_to_name, left_on="hmm_id", right_on="id", how="left")
    .sort(
        ["source", "sample", "ecosystem", "scaffold", "start", "gene_number"],
        descending=[False, False, False, False, False, False]
    )
    .rename({"db": "database"})
    .select(
        ["source", "sample", "ecosystem", "gene", "database", "hmm_id", "name", "evalue", "score", "alignment_type", "coverage_sequence", "coverage_hmm", "keep", "note"]
    )
    .with_columns([
        pl.col("gene").cast(pl.Utf8),
        pl.col("database").cast(pl.Utf8),
        pl.col("hmm_id").cast(pl.Utf8),
        pl.col("evalue").cast(pl.Float64),
        pl.col("score").cast(pl.Float64),
    ])
)

### From DRAM-V

**This took ~30 mins and used up to 300 GB of RAM**.

Also assumes that the`.b6` files of annotations generated from running `DRAM-v.py annotate` with the flag `--keep_tmp_dir` were moved and organized into their own folder `dramv_annotations`, following the same `<SEQUENCE TYPE>/<SAMPLE>` directory structure as `dramv_outputs`. The following files are needed from each DRAM-V run:

* `cazy_results.unprocessed.b6`
* `gene_peptidase_hits.b6`
* `gene_viral_hits.b6`
* `kofam_hmm_results.unprocessed.b6`
* `peptidase_gene_hits.b6`
* `pfam_output.b6`
* `viral_gene_hits.b6`
* `vogdb_results.unprocessed.b6`

In [10]:
import sqlite3

database_description_tables = {
    "dbcan_description": "dbCAN",
    "kegg_description": "KEGG",
    "peptidase_description": "MEROPS",
    "pfam_description": "Pfam",
    "viral_description": "RefSeq Viral",
    "vogdb_description": "VOG",
}

def read_sqlite_table_lazy(db_file, table_name, database_name):
    conn = sqlite3.connect(db_file)

    df = pl.read_database(
        f"SELECT id, description FROM {table_name}",
        conn
    )

    conn.close()

    df = (
        df.rename({"id": "hit_id"})
        .with_columns([
            pl.col("hit_id").cast(pl.Utf8),
            pl.lit(database_name).alias("database")
        ])
    )

    return df.lazy()


db_desc_lazy = []

for table, dbname in database_description_tables.items():
    db_desc_lazy.append(
        read_sqlite_table_lazy(
            "/storage2/databases/DRAM_v1.5.0/description_db.sqlite",
            table,
            dbname
        )
    )

db_desc = (
    pl.concat(db_desc_lazy)
    .select(["database", "hit_id", "description"])
    .unique(subset=["database", "hit_id"])
)

In [11]:
DRAMV_DIR = MAIN_DIR.joinpath("dramv_outputs")

In [12]:
pattern_dramv_raw = "dramv_annotations/*/*/*.b6"
paths_dramv_raw = sorted(MAIN_DIR.glob(pattern_dramv_raw))
patterns_dramv_processed = "*/*/working_dir/final-viral-combined-for-dramv/annotations.tsv"
paths_dramv_processed = sorted(DRAMV_DIR.glob(patterns_dramv_processed))

gene_map = gene_id_mapping.lazy()

db_renames = {
    "kofam": "KEGG",
    "cazy": "dbCAN",
    "pfam": "Pfam",
    "vogdb": "VOG",
    "peptidase": "MEROPS",
    "viral": "RefSeq Viral",
}

import re
def read_hmmsearch_tbl_dram(path: str):
    # Slower than reading into polars directly but robust to formatting issues
    domtable_cols = [
        "target_name", "target_accession", "tlen", 
        "query_name", "query_accession", "qlen",
        "full_evalue", "full_score", "full_bias",
        "domain_n", "domain_of", "c_evalue", "i_evalue",
        "domain_score", "domain_bias",
        "hmm_from", "hmm_to", "ali_from", "ali_to",
        "env_from", "env_to", "acc", "description"
    ]

    domtbl_dict = {x : [] for x in domtable_cols}

    # Need to handle rare cases where the HMMER output contains unexpected line breaks
    rows = []
    buffer = ""

    with open(path) as fh:
        for line in fh:
            if line.strip().startswith("#"):
                continue

            line = line.rstrip()

            if not buffer:
                buffer = line
                continue

            candidate = buffer + " " + line
            parts = re.split(r"\s+", candidate, maxsplit=22)

            if len(parts) >= 22:
                rows.append(candidate)
                buffer = ""
            else:
                buffer = candidate

        if buffer:
            rows.append(buffer)

    for line in rows:
        if line.strip().startswith("#") or line.strip() == "":
            continue
        parts = re.split(r"\s+", line.strip(), maxsplit=22)
        for col, part in zip(domtable_cols, parts):
            if col == "query_name":
                part = part.replace(".hmm", "") # remove .hmm suffix from query names
            domtbl_dict[col].append(part)

    df = pl.DataFrame(domtbl_dict)

    df = (
        df
        .select([
            pl.col("target_name").cast(pl.Utf8).alias("target_name"),
            pl.col("query_name").cast(pl.Utf8).alias("query_name"),
            pl.col("query_accession").cast(pl.Utf8).alias("query_accession"),
            pl.col("full_evalue").alias("evalue"),
            pl.col("full_score").alias("bitscore")
        ])
        .with_columns([
            pl.when(pl.col("query_name") == "-")
            .then(None)
            .otherwise(pl.col("query_name"))
            .alias("query_name"),
            
            pl.when(pl.col("query_accession") == "-")
            .then(None)
            .otherwise(pl.col("query_accession"))
            .alias("query_accession"),

            pl.when(pl.col("evalue") == "-")
            .then(None)
            .otherwise(pl.col("evalue"))
            .cast(pl.Float64)
            .alias("evalue"),

            pl.when(pl.col("bitscore") == "-")
            .then(None)
            .otherwise(pl.col("bitscore"))
            .cast(pl.Float64)
            .alias("bitscore"),
        ])
    )

    return df

def read_mmseqs_b6_tbl(path: str) -> pl.LazyFrame:
    df = pl.read_csv(
        path,
        separator="\t",
        has_header=False,
        comment_prefix="#",
        truncate_ragged_lines=True
    )

    df = (
        df
        .select([
            pl.col("column_1").alias("hit_id"),
            pl.col("column_2").alias("gene"),
            pl.col("column_11").cast(pl.Float64).alias("evalue"),
            pl.col("column_12").cast(pl.Float64).alias("bitscore")
        ])
    )

    return df

# Get "raw" annotations (HMM and MMseqs hits with bitscores and evalues before filtering by DRAM-V)
lazy_dfs_raw = []
for p in paths_dramv_raw:
    parts = p.parts
    try:
        i = parts.index("dramv_annotations")
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
        db = parts[i + 3].split(".")[0]
        if db.startswith("gene_"):
            # skip files with gene queries and db targets, use db queries and gene targets
            continue
        else:
            db = db.split("_")[0]
    except (ValueError, IndexError):
        continue
    
    if db.lower() in ["kofam", "cazy", "vogdb"]:
        print(f"Reading {p} as HMMER output for database {db}")
        df = read_hmmsearch_tbl_dram(str(p))
        df = df.rename({"target_name": "gene", "query_name": "hit_id"}).drop("query_accession")
        # print(f"df: {df.head()}")
    else:
        print(f"Reading {p} as MMseqs2 output for database {db}")
        df = read_mmseqs_b6_tbl(str(p))
        if db == "pfam":
            # Pfam MMseqs2 outputs have gene IDs in the "query_name" column and Pfam accessions in the "hit_id" column, swap them
            df = df.rename({"hit_id": "query_name"})
            df = df.rename({"gene": "hit_id", "query_name": "gene"})
        # print(f"df: {df.head()}")

    df = df.with_columns(pl.col("hit_id").cast(pl.Utf8))

    db = db_renames.get(db.lower(), db)
    
    df = df.with_columns([
        pl.lit(sample).alias("sample"),
        pl.lit(source).alias("source"),
        pl.lit(ecosystem).alias("ecosystem"),
        pl.lit(db).alias("database"),
    ])

    # Join hit IDs to their descriptions from the DRAM description database
    df = df.join(
        db_desc.collect(),
        on=["database", "hit_id"],
        how="left"
    ).rename({"description": "hit_desc"})
    
    df = df.with_columns(
        pl.col("hit_desc").cast(pl.Utf8)
    )

    df = df.join(
        gene_map.collect().select(["sample", "old_gene", "new_gene"]),
        left_on=["gene", "sample"],
        right_on=["old_gene", "sample"],
        how="left"
    ).drop("gene").rename({"new_gene": "gene"})

    # Add sample prefixes to gene names since some scaffold names are identical across samples
    df = df.with_columns(
        pl.concat_str([pl.col("sample"), pl.lit("__"), pl.col("gene")]).alias("gene"),
    )
    df = df.with_columns(
        pl.col("gene").cast(pl.Utf8),
        pl.col("hit_id").cast(pl.Utf8),
        pl.col("database").cast(pl.Utf8),
        pl.col("evalue").cast(pl.Float64),
        pl.col("bitscore").cast(pl.Float64),
        
    )

    lazy_dfs_raw.append(df)

Reading /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_annotations/complete_virus_genomes/virus_genomes_gut/cazy_results.unprocessed.b6 as HMMER output for database cazy
Reading /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_annotations/complete_virus_genomes/virus_genomes_gut/kofam_hmm_results.unprocessed.b6 as HMMER output for database kofam
Reading /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_annotations/complete_virus_genomes/virus_genomes_gut/peptidase_gene_hits.b6 as MMseqs2 output for database peptidase
Reading /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_annotations/complete_virus_genomes/virus_genomes_gut/pfam_output.b6 as MMseqs2 output for database pfam
Reading /storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/dramv_annotations/complete_virus_genomes/virus_genomes_gut/viral_gene_hits.b6 a

In [13]:
dramv_all_annots = pl.concat(lazy_dfs_raw, how="vertical")

### From VIBRANT

Will need access to the table `VIBRANT_names.tsv` that comes packaged with VIBRANT (can be downloaded [here](https://raw.githubusercontent.com/AnantharamanLab/VIBRANT/refs/heads/master/files/VIBRANT_names.tsv)).

In [14]:
VIBRANT_DIR = MAIN_DIR.joinpath("vibrant_outputs")
VIBRANT_HMM_NAMES = Path("/storage2/databases/VIBRANT/files/VIBRANT_names.tsv")

In [15]:
pattern_vibrant_all_annots = "*/*/VIBRANT_genes_reformatted/VIBRANT_HMM_tables_unformatted_genes_reformatted/genes_reformatted_unformatted_*.hmmtbl"
paths_vibrant_all_annots = sorted(VIBRANT_DIR.glob(pattern_vibrant_all_annots))

dfs_vibrant_all_annots = []

vibrant_names = pl.scan_csv(
    VIBRANT_HMM_NAMES,
    separator="\t",
    new_columns=["hit_id", "hit_desc"]
)

def read_hmmsearch_tbl(path: str) -> pl.LazyFrame:
    records = []
    with open(path) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split() # splits on any whitespace run
            if len(parts) < 6:
                continue
            records.append({
                "target_name": parts[0],
                "query_name": parts[2] if parts[2] != "-" else None,
                "query_accession": parts[3] if parts[3] != "-" else None,
                "evalue": float(parts[4]) if parts[4] != "-" else None,
                "bitscore": float(parts[5]) if parts[5] != "-" else None,
            })

    return pl.DataFrame(records, schema={
        "target_name": pl.Utf8,
        "query_name": pl.Utf8,
        "query_accession": pl.Utf8,
        "evalue": pl.Float64,
        "bitscore": pl.Float64,
    }).lazy()

for p in paths_vibrant_all_annots:
    parts = p.parts
    try:
        i = parts.index("vibrant_outputs")
        source = parts[i + 1]
        sample = parts[i + 2]
        ecosystem = sample.split("_")[2] if "_genomes_" in sample else sample.split("_")[0]
        ecosystem = "gut" if ecosystem == "human" else ecosystem
        db = parts[i + 5].split("_")[-1].replace(".hmmtbl", "")  # "VIBRANT_HMM_tables_unformatted_genes_reformatted"
    except (ValueError, IndexError):
        continue

    df = read_hmmsearch_tbl(str(p))
    if db.lower() == "pfam":
        df = (
            df
            .rename(
                {
                    "target_name": "gene",
                    "query_accession": "hit_id",
                }
            )
            .drop("query_name")
        )
    else:
        df = (
            df
            .rename(
                {
                    "target_name": "gene",
                    "query_name": "hit_id",
                }
            )
            .drop("query_accession")
        )
    df = df.with_columns(pl.col("hit_id").cast(pl.Utf8))
    df = df.join(vibrant_names, on="hit_id", how="left")

    df = df.with_columns([
        pl.lit(sample).alias("sample"),
        pl.lit(source).alias("source"),
        pl.lit(ecosystem).alias("ecosystem"),
        pl.lit(db).alias("database"),
    ])

    df = df.join(
        reformat_for_merge(gene_id_mapping).lazy(),
        on=["gene", "sample"],
        how="left"
    )

    # Add sample prefixes to gene names since some scaffold names are identical across samples
    df = df.with_columns(
        pl.concat_str([pl.col("sample"), pl.lit("__"), pl.col("gene")]).alias("gene"),
    )

    df = df.with_columns(
        pl.col("gene").cast(pl.Utf8),
        pl.col("hit_id").cast(pl.Utf8),
        pl.col("database").cast(pl.Utf8),
        pl.col("evalue").cast(pl.Float64),
        pl.col("bitscore").cast(pl.Float64),
    )

    dfs_vibrant_all_annots.append(df)

In [16]:
vibrant_all_annots = pl.concat(dfs_vibrant_all_annots, how="diagonal_relaxed")

## Combine LazyFrames of all annotations from each tool and filter to just AMG genes

In [17]:
AMG_TABLES_DIR = MAIN_DIR.joinpath("amg_tables")
AMG_PREDICTIONS_COMBINED = AMG_TABLES_DIR.joinpath("amg_predictions_combined.parquet")

In [18]:
amg_genes = pl.read_parquet(AMG_PREDICTIONS_COMBINED)

In [19]:
AMG_GENE_LIST = amg_genes.get_column("gene").to_list()

In [20]:
def _normalize_pfam_id(expr: pl.Expr) -> pl.Expr:
    return (
        pl.when(expr.is_not_null() & expr.str.starts_with("PF"))
        .then(expr.str.replace(r"\.\d+$", ""))
        .otherwise(expr)
    )

### Collect the LazyFrame

**This used up to 1500 GB of RAM**

In [21]:
checkamg_block = (
    checkamg_all_annots
    .with_columns([
        pl.lit("CheckAMG").alias("tool"),
        pl.col("hmm_id").alias("hit_id"),
        pl.col("name").alias("hit_desc"),
        pl.col("evalue").cast(pl.Float64).alias("evalue"),
        pl.col("score").cast(pl.Float64).alias("bitscore"),
    ])
    .select(["gene","source","sample","ecosystem","tool","database","hit_id","hit_desc","bitscore","evalue"])
)

dramv_block = (
    dramv_all_annots
    .with_columns([
        pl.lit("DRAMV").alias("tool"),
    ])
    .select(["gene","source","sample","ecosystem","tool","database","hit_id","hit_desc","bitscore","evalue"])
)

vibrant_block = (
    vibrant_all_annots
    .with_columns([
        pl.lit("VIBRANT").alias("tool"),
    ])
    .select(["gene","source","sample","ecosystem","tool","database","hit_id","hit_desc","bitscore","evalue"])
)

In [22]:
def normalize_block(df, tool):
    return (
        df
        .with_columns([
            pl.lit(tool).alias("tool"),
            pl.col("gene").cast(pl.Utf8),
            pl.col("source").cast(pl.Utf8),
            pl.col("sample").cast(pl.Utf8),
            pl.col("ecosystem").cast(pl.Utf8),
            pl.col("database").cast(pl.Utf8),
            pl.col("hit_id").cast(pl.Utf8),
            pl.col("hit_desc").cast(pl.Utf8),
            pl.col("bitscore").cast(pl.Float64),
            pl.col("evalue").cast(pl.Float64)
        ])
        .select([
            "gene","source","sample","ecosystem","tool",
            "database","hit_id","hit_desc","bitscore","evalue"
        ])
    )

In [23]:
def reformat_source_ecosystem_tool(df):
    df = df.with_columns([
        pl.col("ecosystem")
        .str.replace_all("gut", "Human gut", literal=True)
        .str.replace_all("freshwater", "Aquatic", literal=True)
        .str.replace_all("marine", "Aquatic", literal=True)
        .str.replace_all("soil", "Soil", literal=True)
        .str.replace_all("aquatic", "Aquatic", literal=True)
        .alias("ecosystem"),

        pl.col("source")
        .str.replace_all("metagenomes", "Mixed metagenomes", literal=True)
        .str.replace_all("viromes", "Viromes", literal=True)
        .str.replace_all("complete_virus_genomes", "Viral genomes", literal=True)
        .alias("source"),
    ])

    if "tool" in df.columns:
        df = df.with_columns([
            pl.col("tool")
            .str.replace_all("checkamg", "CheckAMG", literal=True)
            .str.replace_all("vibrant", "VIBRANT", literal=True)
            .str.replace_all("dramv", "DRAM-V", literal=True)
            .str.replace_all("DRAMV", "DRAM-V", literal=True)
            .alias("tool"),
        ])

    return df

In [24]:
amg_annots = (
    pl.concat([
        normalize_block(checkamg_block.lazy(),"CheckAMG"),
        normalize_block(dramv_block.lazy(),"DRAMV"),
        normalize_block(vibrant_block.lazy(),"VIBRANT")
    ])
    # Change gene names back and remove version suffices from Pfam IDs
    .with_columns([
        pl.col("gene").str.split("__").list.get(-1).alias("gene"),
        _normalize_pfam_id(pl.col("hit_id").cast(pl.Utf8)).alias("hit_id")

    ])
    # Filter to just the genes that were predicted as AMGs
    .filter(pl.col("gene").is_in(AMG_GENE_LIST))
    # Filter hits with evalue >= 1e-3 and bitscore < 30 to reduce the number of hits
    # but still keep many that weren' in the final outputs
    .filter(
        (pl.col("evalue") <= 1e-3) & (pl.col("bitscore") >= 30)
    )
)

In [25]:
amg_annots = reformat_source_ecosystem_tool(amg_annots)

/tmp/ipykernel_1538212/3985347300.py:18: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if "tool" in df.columns:


In [26]:
checkamg_all_annots, dramv_all_annots, vibrant_all_annots = None, None, None
checkamg_block, dramv_block, vibrant_block = None, None, None

## Label annotations that made it into the final annotations / filtered hits for each tool

### Load the original, combined final results from each tool

Need to load the combined "raw" results from each tool and join by gene, annotation, and database to infer whether an anntoation made it into the final hits for a tool.

#### CheckAMG

In [27]:
checkamg_results_raw = pl.read_parquet(AMG_TABLES_DIR.joinpath("checkamg_results_raw.parquet"))

In [28]:
base_cols = ["source", "sample", "ecosystem", "gene"]

dfs_checkamg_raw = []

# KEGG
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["KEGG KO", "KEGG KO Name"])
    .rename({"KEGG KO": "id", "KEGG KO Name": "name"})
    .with_columns(pl.lit("KEGG").alias("db"))
)

# FOAM
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["FOAM ID", "FOAM Annotation"])
    .rename({"FOAM ID": "id", "FOAM Annotation": "name"})
    .with_columns(pl.lit("FOAM").alias("db"))
)

# Pfam
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["Pfam Accession", "Pfam Name"])
    .rename({"Pfam Accession": "id", "Pfam Name": "name"})
    .with_columns(pl.lit("Pfam").alias("db"))
)

# CAZy
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["CAZy Family", "CAZy Activities"])
    .rename({"CAZy Family": "id", "CAZy Activities": "name"})
    .with_columns(pl.lit("CAZy").alias("db"))
)

# METABOLIC
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["METABOLIC db ID", "METABOLIC Annotation"])
    .rename({"METABOLIC db ID": "id", "METABOLIC Annotation": "name"})
    .with_columns(pl.lit("METABOLIC").alias("db"))
)

# CAMPER
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["CAMPER ID", "CAMPER Annotation"])
    .rename({"CAMPER ID": "id", "CAMPER Annotation": "name"})
    .with_columns(pl.lit("CAMPER").alias("db"))
)

# PHROG
dfs_checkamg_raw.append(
    checkamg_results_raw
    .select(base_cols + ["PHROG Number", "PHROG Annotation"])
    .rename({"PHROG Number": "id", "PHROG Annotation": "name"})
    .with_columns(pl.lit("PHROG").alias("db"))
)

# combine + drop null ids
checkamg_results_raw_db_long = (
    pl.concat(dfs_checkamg_raw, how="vertical_relaxed")
    .filter(pl.col("id").is_not_null())
    .with_columns([
        _normalize_pfam_id(pl.col("id").cast(pl.Utf8)).alias("id"),
        pl.lit("CheckAMG").alias("tool")
        ])
)

checkamg_results_raw_db_long = reformat_source_ecosystem_tool(checkamg_results_raw_db_long)

In [29]:
checkamg_results_raw_db_long

source,sample,ecosystem,gene,id,name,db,tool
str,str,str,str,str,str,str,str
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_1""","""K07497""","""K07497; putative transposase""","""KEGG""","""CheckAMG"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_4""","""K06400""","""spoIVCA; site-specific DNA recombinase""","""KEGG""","""CheckAMG"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_7""","""K01356""","""lexA; repressor LexA [EC:3.4.21.88]""","""KEGG""","""CheckAMG"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_18""","""K03111""","""ssb; single-strand DNA-binding protein""","""KEGG""","""CheckAMG"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_36""","""K01358""","""clpP, CLPP; ATP-dependent Clp protease, protease subunit [EC:3.4.21.92]""","""KEGG""","""CheckAMG"""
…,…,…,…,…,…,…,…
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_9_c1_5""","""phrog_12179""","""""","""PHROG""","""CheckAMG"""
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_9_c1_8""","""phrog_2392""","""ABC transporter""","""PHROG""","""CheckAMG"""
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_9_c1_15""","""phrog_5687""","""""","""PHROG""","""CheckAMG"""


#### DRAM-V

In [30]:
dramv_results_raw = pl.read_parquet(AMG_TABLES_DIR.joinpath("dramv_results_raw.parquet"))

In [31]:
dramv_results_raw = (
    dramv_results_raw
    .select([
        "source", "sample", "ecosystem", "gene",
        "gene_id", "gene_description",
    ])
    .rename({"gene_id": "id", "gene_description": "name"})
    .with_columns([
        _normalize_pfam_id(pl.col("id").cast(pl.Utf8)).alias("id"),
        pl.lit("DRAM-V").alias("tool")
    ])
)
dramv_results_raw = reformat_source_ecosystem_tool(dramv_results_raw)
dramv_results_raw

source,sample,ecosystem,gene,id,name,tool
str,str,str,str,str,str,str
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_2""","""PF04896""","""Ammonia monooxygenase/methane monooxygenase, subunit C""","""DRAM-V"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_2""","""PF00317""","""nrdA/nrdB; ribonucleotide reductase (RNR)""","""DRAM-V"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_5""","""PF00262""","""Calreticulin family""","""DRAM-V"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_5""","""PF02347""","""Glycine cleavage system P-protein""","""DRAM-V"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2515154176_000011_2515154176_2515168409_29861-65525_6""","""PF01832""","""Mannosyl-glycoprotein endo-beta-N-acetylglucosaminidase""","""DRAM-V"""
…,…,…,…,…,…,…
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_9_c1_33""","""PF00485""","""Phosphoribulokinase / Uridine kinase family""","""DRAM-V"""
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_9_c1_33""","""PF02397""","""Bacterial sugar transferase""","""DRAM-V"""
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_9_c1_33""","""PF00266""","""Aminotransferase class-V""","""DRAM-V"""


#### VIBRANT

In [32]:
vibrant_results_raw = pl.read_parquet(AMG_TABLES_DIR.joinpath("vibrant_results_raw.parquet"))

In [33]:
dfs_vibrant_raw = []

# KEGG (AMG KO)
dfs_vibrant_raw.append(
    vibrant_results_raw
    .select(base_cols + ["AMG KO", "AMG KO name"])
    .rename({"AMG KO": "id", "AMG KO name": "name"})
    .with_columns(pl.lit("KEGG").alias("db"))
)

# Pfam
dfs_vibrant_raw.append(
    vibrant_results_raw
    .select(base_cols + ["Pfam", "Pfam name"])
    .rename({"Pfam": "id", "Pfam name": "name"})
    .with_columns(pl.lit("Pfam").alias("db"))
)

vibrant_results_raw_db_long = (
    pl.concat(dfs_vibrant_raw, how="vertical_relaxed")
    .filter(pl.col("id").is_not_null() & (pl.col("id") != ""))
    .with_columns([
        _normalize_pfam_id(pl.col("id").cast(pl.Utf8)).alias("id"),
        pl.lit("VIBRANT").alias("tool")
    ])
)

vibrant_results_raw_db_long = reformat_source_ecosystem_tool(vibrant_results_raw_db_long)

In [34]:
vibrant_results_raw_db_long

source,sample,ecosystem,gene,id,name,db,tool
str,str,str,str,str,str,str,str
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2546825507_000002_2546825507_2546825919_26141-66258_2""","""K01772""","""hemH, FECH; protoporphyrin/coproporphyrin ferrochelatase [EC:4.99.1.1 4.99.1.9]""","""KEGG""","""VIBRANT"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2693429676_000016_2693429676_2693458520_27402-60603_10""","""K00558""","""DNMT1, dcm; DNA (cytosine-5)-methyltransferase 1 [EC:2.1.1.37]""","""KEGG""","""VIBRANT"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2718218227_000005_2718218227_2718228826_2038485-2104644_74""","""K21140""","""mec; [CysO sulfur-carrier protein]-S-L-cysteine hydrolase [EC:3.13.1.6]""","""KEGG""","""VIBRANT"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2839645069_000001_2839645069_2839645076_48518-102011_47""","""K00390""","""cysH; phosphoadenosine phosphosulfate reductase [EC:1.8.4.8 1.8.4.10]""","""KEGG""","""VIBRANT"""
"""Viral genomes""","""virus_genomes_gut""","""Human gut""","""IMGVR_UViG_2904879368_000003_2904879368_2904879379_176713-228057_32""","""K00558""","""DNMT1, dcm; DNA (cytosine-5)-methyltransferase 1 [EC:2.1.1.37]""","""KEGG""","""VIBRANT"""
…,…,…,…,…,…,…,…
"""Viromes""","""soil_T42_15_2_44""","""Soil""","""scaffold_9493_c1_8""","""PF01259""","""SAICAR synthetase""","""Pfam""","""VIBRANT"""
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_22_c1_40""","""PF00877""","""NlpC/P60 family""","""Pfam""","""VIBRANT"""
"""Viromes""","""soil_T42_15_3_45""","""Soil""","""scaffold_473_c1_10""","""PF00145""","""C-5 cytosine-specific DNA methylase""","""Pfam""","""VIBRANT"""


### Join with AMG anntoations and label the filtered/final annots

In [35]:
amg_annots_checkamg_labeled = (
    amg_annots
    .filter(pl.col("tool") == "CheckAMG")
    .join(
        (
            checkamg_results_raw_db_long
            .select(["source", "sample", "ecosystem", "gene", "db", "id"])
            .with_columns(pl.lit(True).alias("final_annot"))
            .lazy()
        ),
        left_on=["source", "sample", "ecosystem", "gene", "database", "hit_id"],
        right_on=["source", "sample", "ecosystem", "gene", "db", "id"],
        how="left"
    )
    .unique()
    .sort(["source", "sample", "ecosystem", "gene", "evalue"], descending=[False, False, False, False, False])
    .with_columns(pl.col("final_annot").fill_null(False))
)

In [36]:
amg_annots_dramv_labeled = (
    amg_annots
    .filter(pl.col("tool") == "DRAM-V")
    .join(
        (
            dramv_results_raw
            .select(["source", "sample", "ecosystem", "gene", "id"])
            .with_columns(pl.lit(True).alias("final_annot"))
            .lazy()
        ),
        left_on=["source", "sample", "ecosystem", "gene", "hit_id"],
        right_on=["source", "sample", "ecosystem", "gene", "id"],
        how="left"
    )
    .unique()
    .sort(["source", "sample", "ecosystem", "gene", "evalue"], descending=[False, False, False, False, False])
    .with_columns(pl.col("final_annot").fill_null(False))
)

In [37]:
amg_annots_vibrant_labeled = (
    amg_annots
    .filter(pl.col("tool") == "VIBRANT")
    .join(
        (
            vibrant_results_raw_db_long
            .select(["source", "sample", "ecosystem", "gene", "id", "db"])
            .with_columns(pl.lit(True).alias("final_annot"))
            .lazy()
        ),
        left_on=["source", "sample", "ecosystem", "gene", "hit_id", "database"],
        right_on=["source", "sample", "ecosystem", "gene", "id", "db"],
        how="left"
    )
    .unique()
    .sort(["source", "sample", "ecosystem", "gene", "evalue"], descending=[False, False, False, False, False])
    .with_columns(pl.col("final_annot").fill_null(False))
)

The `.collect()` operation will use **up to ~1500 GB of RAM**. To avoid this, `.collect()` can be removed and the annotation table can be written directly using `.sink_parquet()`.

In [38]:
amg_annots_df_labeled = (
    pl.concat([
        amg_annots_checkamg_labeled,
        amg_annots_dramv_labeled,
        amg_annots_vibrant_labeled
    ])
    .collect()
)

In [39]:
amg_annots_df_labeled

gene,source,sample,ecosystem,tool,database,hit_id,hit_desc,bitscore,evalue,final_annot
str,str,str,str,str,str,str,str,f64,f64,bool
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""KEGG""","""K04708""","""KDSR, kdsr; 3-dehydrosphinganine reductase [EC:1.1.1.102]""",297.572968,2.4026e-87,true
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""KEGG""","""K07124""","""K07124; uncharacterized protein""",185.187241,3.0093e-53,false
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""CAMPER""","""K07535""","""2-hydroxycyclohexanecarboxyl-CoA dehydrogenase [EC:1.1.1.-]""",178.950897,2.1448e-51,true
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""CAMPER""","""K07535""","""badH; 2-hydroxycyclohexanecarboxyl-CoA dehydrogenase [EC:1.1.1.-]""",178.950897,2.1448e-51,true
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""Pfam""","""PF00106""","""short chain dehydrogenase""",177.693161,2.3708e-51,true
…,…,…,…,…,…,…,…,…,…,…
"""scaffold_9_c1_8""","""Viromes""","""soil_T42_15_3_45""","""Soil""","""VIBRANT""","""KEGG""","""K02071""","""metN; D-methionine transport system ATP-binding protein""",41.1,2.1000e-11,false
"""scaffold_9_c1_8""","""Viromes""","""soil_T42_15_3_45""","""Soil""","""VIBRANT""","""KEGG""","""K10558""","""lsrA, ego; AI-2 transport system ATP-binding protein""",40.3,2.5000e-11,false
"""scaffold_9_c1_8""","""Viromes""","""soil_T42_15_3_45""","""Soil""","""VIBRANT""","""KEGG""","""K10021""","""occP, nocP; octopine/nopaline transport system ATP-binding protein [EC:7.4.2.1]""",40.4,2.8000e-11,false


In [40]:
AMG_ANNOTATIONS_OUTPUT = AMG_PREDICTIONS_COMBINED.parent.joinpath("amg_all_annotations.parquet")

In [41]:
amg_annots_df_labeled.write_parquet(AMG_ANNOTATIONS_OUTPUT)